# 06 - build feature matrices, combos, and the full benchmark grid

Assembles every representation as a `mutant`-keyed block over the DMS label spine, benchmarks each alone and in curated complementary combinations through nested CV (inner grid search per outer fold), pooled out-of-fold AUROC, bootstrap 95% CIs, across random, modulo, and contiguous splits.

Cells are named directly, no abstractions: `<Classifier>_<ExactFeatureSet>_<split>`. Classifiers: Logistic Regression, Random Forest, XGBoost, SVM. Contiguous is the headline.

Leakage guard: every feature matrix asserts the sealed 13 are absent before any training.

In [ ]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/".projectroot").exists())
sys.path.insert(0, str(_root))
from paths import RAW, INTERIM, PROCESSED, FIGURES, TABLES, PROJECT_ROOT
SRC = _root / "src"


In [ ]:
import subprocess, sys
def run_in(env, script, *args):
    """Run src/<script> in conda env <env>; stream output; raise on failure."""
    cmd = ["conda","run","--no-capture-output","-n",env,"python",str(SRC/script), *map(str,args)]
    print(">>", " ".join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0: raise RuntimeError(f"{script} failed in env {env} (rc={r.returncode})")


## Optional, before a full run: per-variant ESMFold delta coverage

The predicted-structure tier always has WT-only structural features (`ESMFold_struct`, full coverage, from notebook 05). The richer per-variant delta block (`ESMFold_delta`, dpLDDT/dSASA per mutation, not just per position) only exists once `src/colab_esmfold/09_colab_esmfold_extraction.ipynb` has actually been run, which is a GPU job over all 4,770 variants (hours on a real GPU, much longer without one; see its own header for device notes). It's not required to run the grid below. Without it, the predicted-structure tier just falls back to WT-only features, same as always. Run it whenever you want the full-coverage delta arm included, then re-run `build_struct_features.py` (triggered automatically inside `run_benchmark.py` below) before this cell.

In [ ]:
delta_csv = INTERIM / "esmfold_variant_features.csv"
if delta_csv.exists():
    import pandas as pd
    n = len(pd.read_csv(delta_csv))
    print(f"ESMFold delta coverage found: {n}/4770 variants in {delta_csv.name}")
else:
    print("no ESMFold delta coverage yet -- predicted-structure tier will use WT-only features.")
    print("run src/colab_esmfold/09_colab_esmfold_extraction.ipynb first for full delta coverage.")


## Run the benchmark as two separate arms, plus wave-2 additions

Per project design, the grid runs as two separately-labelled arms on the identical 4,770-variant pool, so every AUROC is directly comparable:

1. `sequence`: all PLM tiers (ESM-2, ESM-1v, ESM C x Rep1/2/3/4 plus combos) and PLM-free controls (identity one-hot, physicochemical). ESMFold excluded. Writes `plm_benchmark_sequence_only.csv`.
2. `esmfold`: the predicted-structure tier. WT position-mapped structural features alone and combined with PLM/physicochemical features, plus the identity/physicochemical floors for reference. Writes `plm_benchmark_esmfold.csv`.

Wave-2 then appends the feature-design-doc representations the primary grid missed (global surprisal, thresholded binary fingerprint, sliding-window surprisal, sensitive-residue surprisal, physicochemical sliding-window), plus mixed-family multi-way combos (surprisal, embedding, physicochemical, and structure together, including cross-PLM mixes).

Each arm checkpoints per cell (`benchmark_partial_*.csv`), so an interrupted run resumes where it stopped.

In [ ]:
# Arm 1: sequence-only (PLM tiers + PLM-free controls; ESMFold excluded)
run_in('betalactam-v2','run_benchmark.py','sequence')


In [ ]:
# Arm 2: ESMFold predicted-structure tier (position-mapped, all 4,770; + PLM/physchem combos)
run_in('betalactam-v2','run_benchmark.py','esmfold')


### Wave-2 additions (design-doc representations and mixed-family combos)

`run_benchmark_wave2_defs.py` enumerates the 38 additional sets. They append into the same two arm tables: sequence-tier sets go to the sequence CSV, predicted-structure sets go to the esmfold CSV.

In [ ]:
# Wave-2: build + benchmark the 38 additional feature sets, merge into the arm tables
run_in('betalactam-v2','run_benchmark_wave2.py')


## Inspect results: contiguous split (the honest headline)

In [ ]:
import pandas as pd
seq = pd.read_csv(TABLES/'plm_benchmark_sequence_only.csv'); seq['arm']='sequence'
esm = pd.read_csv(TABLES/'plm_benchmark_esmfold.csv');       esm['arm']='esmfold'
res = pd.concat([seq,esm], ignore_index=True)
cont = res[res.split=='contiguous'].sort_values('auroc',ascending=False)
print('SEQUENCE-ONLY arm, contiguous top 15:')
print(cont[cont.arm=='sequence'].head(15)[['classifier','feature_set','auroc','ci_lo','ci_hi']].to_string(index=False))
print()
print('ESMFOLD arm, contiguous top 10:')
print(cont[cont.arm=='esmfold'].head(10)[['classifier','feature_set','auroc','ci_lo','ci_hi']].to_string(index=False))
print()
floor = cont[cont.feature_set=='Identity one-hot'].auroc.max()
print(f'Identity one-hot floor (best classifier, contiguous): {floor:.4f}')
